In [10]:
import numpy as np
import pandas as pd 
import scipy.stats
import gudhi as gd
import networkx as nx 
import random
from tqdm import tqdm

In [11]:
#extended version
def compute_persistent_homology(graph, filtration):
    """
    Compute the persistent homology of a graph with a given filtration.

    :param graph: A networkx graph.
    :param filtration: A list of node filtration values, corresponding to each node in the graph.
    :return: Persistent homology of dimension one.
    """
    # Create simplex tree
    st = gd.SimplexTree()

    # Add vertices with filtration values
    for i, node in enumerate(graph.nodes):
        st.insert([node], filtration=filtration[i])


    # Add edges
    for edge in graph.edges:
        st.insert(list(edge), filtration=max(filtration[list(graph.nodes()).index(edge[0])], filtration[list(graph.nodes()).index(edge[1])]))
    # Expand to clique complex
    st.extend_filtration()

    #romve null cycles
    persistence = st.extended_persistence(min_persistence=1e-5)
    tmp = []
    tmp.extend(persistence[0])
    tmp.extend(persistence[1])
    tmp.extend(persistence[2])
    tmp.extend(persistence[3])
    persistence = tmp

    #if birth > death change them
    tmp = []
    for gen in persistence:
        tmp.append((gen[0],(min(gen[1][0],gen[1][1]),max(gen[1][0],gen[1][1])) ))

    persistence = tmp

    #Filter for dimension one homology
    dim1_res = []
    for gen in persistence:
        if gen[0] == 1:
            if gen[1][0] != gen[1][1]:
                dim1_res.append(gen)
    return dim1_res


## unweighted - given_th

#### Complex

In [12]:
network_list = [
    'conf','email_eu','hospital','school','work'
]

In [13]:
for network_name in network_list:
    #loading simulations and graphs
    sims = pd.read_csv(f'../results/unweighted/given_th_n=200/unweighted_complex_{network_name}.csv')
    G = nx.read_graphml(f'../networks/unweighted/G_unweighted_{network_name}.graphml')

    #preprocessing
    # Create a mapping from current node names (str) to integers
    mapping = {node: int(float(node)) for node in G.nodes()}
    # Relabel the nodes in the graph using the mapping
    G = nx.relabel_nodes(G, mapping)
    
    #order of infection 
    order = sims.drop(['Unnamed: 0','theta','seed','q'],axis=1)
    order_prl = order.copy(deep=True)
    order = order.apply(lambda row: row.fillna(row.max() + 1), axis=1)
    
    #EPH for each simulations
    EPH = []
    corr = [] 
    for row_index in tqdm(range(len(order))):
        #defining filtration
        filt = list(-order.iloc[row_index].values)
        #compute EPH
        persistent_homology = compute_persistent_homology(graph=G,filtration=filt)

        #lifetime of genrators 
        life_set = []
        for gen in persistent_homology:
            life_set.append(gen[1][1] - gen[1][0])
        EPH.append(np.nanmean(life_set))
        
        #PRL
        deg_list = list(dict(G.degree).values())
        order_curr = list(order_prl.iloc[row_index].values)
        
        try:
            rho, p_value = scipy.stats.spearmanr(deg_list, order_curr,nan_policy='omit')
        except:
            print('PRL method could\'nt find rho!!')
            rho = 0
            
        corr.append(rho)
        
    sims['EPH'] = EPH
    sims['corr'] = corr
    sims.to_csv(f'../results/unweighted/given_th_n=200/unweighted_complex_{network_name}_EPH.csv')      

100%|██████████| 1000/1000 [00:02<00:00, 383.10it/s]


#### Simple

In [14]:
for network_name in network_list:
    #loading simulations and graphs
    sims = pd.read_csv(f'../results/unweighted/given_th_n=200/unweighted_simple_{network_name}.csv')
    G = nx.read_graphml(f'../networks/unweighted/G_unweighted_{network_name}.graphml')

    #preprocessing
    # Create a mapping from current node names (str) to integers
    mapping = {node: int(float(node)) for node in G.nodes()}
    # Relabel the nodes in the graph using the mapping
    G = nx.relabel_nodes(G, mapping)
    
    #order of infection 
    order = sims.drop(['Unnamed: 0','betha','seed'],axis=1)
    order_prl = order.copy(deep=True)
    order = order.apply(lambda row: row.fillna(row.max() + 1), axis=1)
    
    #EPH for each simulations
    EPH = []
    corr = [] 
    for row_index in tqdm(range(len(order))):
        #defining filtration
        filt = list(-order.iloc[row_index].values)
        #compute EPH
        persistent_homology = compute_persistent_homology(graph=G,filtration=filt)

        #lifetime of genrators 
        life_set = []
        for gen in persistent_homology:
            life_set.append(gen[1][1] - gen[1][0])
        EPH.append(np.nanmean(life_set))
        
        #PRL
        deg_list = list(dict(G.degree).values())
        order_curr = list(order_prl.iloc[row_index].values)
        
        try:
            rho, p_value = scipy.stats.spearmanr(deg_list, order_curr,nan_policy='omit')
        except:
            print('PRL method could\'nt find rho!!')
            rho = 0
            
        corr.append(rho)
        
    sims['EPH'] = EPH
    sims['corr'] = corr
    sims.to_csv(f'../results/unweighted/given_th_n=200/unweighted_simple_{network_name}_EPH.csv')        

100%|██████████| 200/200 [00:00<00:00, 389.59it/s]


In [16]:
1

1